# GISPR Module 4 — Vector Operations: Buffers, Clip & Overlay, and Spatial Joins

**Course:** GIS Spatial Analysis with Python and R (GISPR)
**Module:** 4 — Vector Operations
**Builds on:** Module 3 (loading vector data, CRS, attribute cleaning, tabular → spatial, GeoPackage export)
**Feeds into:** Module 5 (raster data) and the vector + raster pipelines in Modules 9–11

---

### Tonight's Exercise

Last week you pulled four salmon datasets into one harmonized, EPSG:2926 package. This week the questions get spatial:

> *Which publicly owned parcels around Issaquah sit within a riparian buffer of a fish-bearing stream — in the watersheds where Chinook were actually observed — and how many acres does each owner type hold?*

Answering that takes all three operations in this module: a **spatial join** (which watershed is each observation in?), a **clip** and **overlay** (streams inside those watersheds, parcel area inside the buffer) and a **buffer** (the riparian zone). You've done each of these in ArcGIS Pro. Tonight you chain them in code, so the whole analysis re-runs in seconds when the parcel layer or the buffer rule changes.

### What you'll accomplish this module

| # | Learning outcome | Where |
|---|---|---|
| 1 | Buffer a vector layer at a meaningful distance after confirming CRS **and units** | Section 2 |
| 2 | Clip one layer to another polygon boundary | Section 3 |
| 3 | Run overlay operations (intersection, difference, union) | Section 3 |
| 4 | Spatial-join attributes from a polygon layer to a point layer, including nearest-feature joins | Section 4 |
| 5 | Chain all three operations into a complete, reproducible workflow | Section 5 |
| 6 | Identify and fix the three most common spatial operation errors | Section 6 |

### Kernel reminder
- 🔵 **`[R]`** cells — switch kernel to **R** before running
- 🟢 **`[Python]`** cells — switch kernel to **Python 3** (or your cloned ArcGIS Pro env) before running
- 🟢 **`[Python / ArcPy]`** cells — run in ArcGIS Pro's Notebook or your cloned `arcgispro-py3` env. Anywhere `arcpy` isn't installed they print a note and skip, and the GeoPandas cells that follow do the same work.

### A note on units — this matters more tonight than last week

Every layer is in **EPSG:2926 — NAD83(HARN) / Washington North, US survey feet**, the Module 3 course CRS. Buffer distances and areas therefore come out in **feet** and **square feet**: `buffer(150)` is 150 ft (≈ 46 m), not 150 m. Divide square feet by **43,560** to get acres.

### Data used in this notebook

We reuse the **Module 3 data bundle** (`GISPR_Module3_Data/data`). The analysis starts from the two File Geodatabases, the way agency data usually arrives:

| Layer | Source | Features | Used for |
|---|---|---|---|
| `watersheds` | `salmon_watersheds.gdb` | 15 WBD HUC-12s, WRIA 8 | Clip mask, spatial join target |
| `streams` | `salmon_watersheds.gdb` | 464 NHDPlus V2 flowlines | Buffer source, clip input |
| `wria8_boundary` | `salmon_watersheds.gdb` | 1 | Map context |
| `Parcels` | `parcels.gdb` | 451 public parcels around Issaquah | Overlay input (riparian acres) |
| Salmon observations | **GISPR Salmon Observations** hosted layer (or `salmon_observations_hosted_backup.gpkg`) | 60 points | Spatial join source |
| Partner agency observations | `salmon_obs_partner_agency.geojson` | 25 points, EPSG:4326 | CRS-mismatch demo, left vs. inner joins |
| Water-quality sites | `monitoring_sites.csv` | 800 points (lon/lat) | Nearest-feature join, Guided Lab |

> Reminder from Module 3: the watersheds, streams and parcels are real; the salmon observations and water-quality values are **synthetic** (real locations on real flowlines, invented attributes). Use them for learning, not for real decisions.

> **The goal isn't to memorize syntax.** It's to build the habit: load → check CRS → operate → verify → export → commit.

---
**Sections in this notebook:**
1. Setup — Load the Module 3 Geodatabases
2. Buffers — Creating Distance Zones
3. Clip & Overlay — Cutting and Combining Layers
4. Spatial Joins — Location as the Join Key
5. Complete Workflow — Chaining All Three Operations
6. Common Errors and How to Fix Them
7. Extra Credit — Apply Concepts to a new scenario
8. Extra Credit - Challenge Extensions
9. Homework Deliverables

---
## Section 1 — Setup: Load the Module 3 Geodatabases

Point `DATA` at the same unzipped `GISPR_Module3_Data/data` folder you used last week. If you left it in the Module 3 folder, the cell finds it on its own. Outputs from this module go in `outputs/` next to this notebook.

The setup cell reads every layer the notebook needs **once** — straight from `salmon_watersheds.gdb` and `parcels.gdb` with GDAL's OpenFileGDB driver (no ArcGIS license needed) — plus the salmon observations from the hosted layer, falling back to the offline backup just as in Module 3. Later cells reuse these objects, so run the setup cell for each kernel first.

Two small bits of Module 3-style cleaning happen on load, because they matter for tonight's operations:
- **Fish-bearing streams.** We define them as NHDPlus flowlines with `FTYPE == "Stream/River"` and `STREAM_ORDER >= 2`, which leaves out lake artificial paths, coastline, pipelines and ditches. (The `FTYPE` column even has `"Artifical Path"` misspelled on most rows. Filtering on the value you *want* avoids that problem.)
- **Parcel keys.** Five `PARCEL_ID`s are null, so we build a `parcel_key` that is never null. We'll compute parcel area from geometry instead of trusting the text `AREA_SQFT` column.

### This cell will run here but be sure to run it in your ArcGIS Pro notebook so all the variables are definied!

🟢 **Python**

In [ ]:
# [Python] Section 1 — Setup: paths, course CRS, and load layers from the Module 3 geodatabases
from pathlib import Path
import warnings
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt

DATA = Path('your/data/path/on/your/machine')   # the unzipped GISPR_Module3_Data/data folder from Module 3
if not (DATA / 'salmon_watersheds.gdb').exists():  # not set? look in the usual places
    for gdb in [Path('data'), Path('../Module 3/GISPR_Module3_Data/data'),
                 Path('../GISPR_Module3_Data/data'), Path('GISPR_Module3_Data/data')]:
        if (gdb / 'salmon_watersheds.gdb').exists():
            DATA = gdb
            break
OUT = Path('outputs'); OUT.mkdir(exist_ok=True)  # Module 4 outputs, next to this notebook
COURSE_EPSG  = 2926          # NAD83(HARN) / Washington North — US survey feet
SQFT_PER_ACRE = 43_560
BUFFER_FT    = 150           # riparian buffer distance used throughout (feet!)

# Paste the REST URL of the "GISPR Salmon Observations" hosted layer (posted on Canvas)
SALMON_OBS_URL = 'https://services2.arcgis.com/xv4COPn03HHYL16S/arcgis/rest/services/GISPR_Salmon_Observations/FeatureServer'

WS_GDB     = DATA / 'salmon_watersheds.gdb'
PARCEL_GDB = DATA / 'parcels.gdb'
assert WS_GDB.exists() and PARCEL_GDB.exists(), (
    f"Can't find the Module 3 geodatabases in {DATA.resolve()} — set DATA to your GISPR_Module3_Data/data folder.")
print(f"✅  Module 3 data folder: {DATA.resolve()}")

def list_layers(path):
    # Layer names in a GDB / GeoPackage — works with or without pyogrio installed (Module 3 helper)
    try:
        return list(gpd.list_layers(path)['name'])   # geopandas >= 1.0 with pyogrio
    except (ImportError, AttributeError):
        import fiona                                  # ships with the ArcGIS Pro env
        return fiona.listlayers(str(path))

print("salmon_watersheds.gdb:", list_layers(WS_GDB))
print("parcels.gdb:          ", list_layers(PARCEL_GDB))

# ── Read the feature classes ────────────────────────────────────────────────
watersheds = gpd.read_file(WS_GDB, layer='watersheds')
streams    = gpd.read_file(WS_GDB, layer='streams')
wria8      = gpd.read_file(WS_GDB, layer='wria8_boundary')
parcels    = gpd.read_file(PARCEL_GDB, layer='Parcels')

# ── Salmon observations: hosted layer first, offline backup if that fails (as in Module 3) ──
def load_salmon_obs():
    try:
        from arcgis.gis import GIS
        from arcgis.features import FeatureLayer
        sdf = FeatureLayer(SALMON_OBS_URL, gis=GIS('home')).query(out_sr=COURSE_EPSG).sdf
        geom = sdf.spatial.name
        gdf = gpd.GeoDataFrame(sdf.drop(columns=geom),
                               geometry=gpd.GeoSeries.from_wkt(sdf[geom].apply(lambda g: g.WKT)),
                               crs=COURSE_EPSG)
        print("Salmon observations: read from the hosted feature layer")
        return gdf
    except Exception as e:
        print(f"Salmon observations: hosted layer unavailable ({type(e).__name__}) — using the offline backup")
        return gpd.read_file(DATA / 'salmon_observations_hosted_backup.gpkg')

salmon_obs = load_salmon_obs()

# ── Defensive CRS check (Module 3, Section 3c): everything into the course CRS ──
layers = {'watersheds': watersheds, 'streams': streams, 'wria8': wria8,
          'parcels': parcels, 'salmon_obs': salmon_obs}
for name, gdf in layers.items():
    if gdf.crs.to_epsg() != COURSE_EPSG:
        print(f"  reprojecting {name}: EPSG:{gdf.crs.to_epsg()} → {COURSE_EPSG}")
        layers[name] = gdf.to_crs(COURSE_EPSG)
watersheds, streams, wria8, parcels, salmon_obs = layers.values()

# ── Light cleaning for tonight's operations ────────────────────────────────
fish_streams = streams[(streams['FTYPE'] == 'Stream/River') & (streams['STREAM_ORDER'] >= 2)].copy()
parcels['parcel_key'] = parcels['PARCEL_ID'].fillna(pd.Series('NOID-' + parcels.index.astype(str), index=parcels.index))
parcels['area_acres'] = (parcels.geometry.area / SQFT_PER_ACRE).round(3)   # from geometry, not AREA_SQFT text
issaquah_ws = watersheds[watersheds['WATERSHED_NAME'] == 'Issaquah Creek']

print()
for name, gdf in {**dict(zip(layers, (watersheds, streams, wria8, parcels, salmon_obs))),
                  'fish_streams': fish_streams}.items():
    print(f"  {name:<13} {len(gdf):>4} features | {gdf.geom_type.iloc[0]:<16} | EPSG:{gdf.crs.to_epsg()}")

🔵 **`R`**

In [ ]:
# [R] Section 1 — Setup: paths, course CRS, and load layers from the Module 3 geodatabases
library(sf)
library(dplyr, warn.conflicts = FALSE)

DATA <- "your/data/path/on/your/machine"         # the unzipped GISPR_Module3_Data/data folder from Module 3
if (!dir.exists(file.path(DATA, "salmon_watersheds.gdb"))) {   # not set? look in the usual places
  for (cand in c("data", "../Module 3/GISPR_Module3_Data/data", "../GISPR_Module3_Data/data",
                 "GISPR_Module3_Data/data")) {
    if (dir.exists(file.path(cand, "salmon_watersheds.gdb"))) { DATA <- cand; break }
  }
}
OUT <- "outputs"; dir.create(OUT, showWarnings = FALSE)
COURSE_EPSG   <- 2926        # NAD83(HARN) / Washington North — US survey feet
SQFT_PER_ACRE <- 43560
BUFFER_FT     <- 150         # riparian buffer distance (feet!)

WS_GDB     <- file.path(DATA, "salmon_watersheds.gdb")
PARCEL_GDB <- file.path(DATA, "parcels.gdb")
stopifnot("Set DATA to your GISPR_Module3_Data/data folder" = dir.exists(WS_GDB) && dir.exists(PARCEL_GDB))
cat("✅  Module 3 data folder:", normalizePath(DATA), "\n")
print(st_layers(WS_GDB)$name)
print(st_layers(PARCEL_GDB)$name)

# ── Read the feature classes ────────────────────────────────────────────────
watersheds <- st_read(WS_GDB, layer = "watersheds",     quiet = TRUE)
streams    <- st_read(WS_GDB, layer = "streams",        quiet = TRUE)
wria8      <- st_read(WS_GDB, layer = "wria8_boundary", quiet = TRUE)
parcels    <- st_read(PARCEL_GDB, layer = "Parcels",    quiet = TRUE)

# ── Salmon observations: offline backup of the hosted layer ────────────────
# (To read the live hosted layer instead, run the arc_signin() / arcgislayers cells from Module 3:
#  salmon_obs <- arcgislayers::arc_read(SALMON_OBS_URL) )
salmon_obs <- st_read(file.path(DATA, "salmon_observations_hosted_backup.gpkg"), quiet = TRUE)

# ── Defensive CRS check: everything into the course CRS ────────────────────
# (also renames the geometry column — "SHAPE" in a File GDB — to "geometry" so every layer matches)
to_course <- function(x) {
  if (!isTRUE(st_crs(x)$epsg == COURSE_EPSG)) x <- st_transform(x, COURSE_EPSG)
  st_geometry(x) <- "geometry"
  st_agr(x) <- "constant"   # attributes describe whole features — silences st_intersection()'s warning
  x
}
watersheds <- to_course(watersheds); streams <- to_course(streams); wria8 <- to_course(wria8)
parcels    <- to_course(parcels);    salmon_obs <- to_course(salmon_obs)

# ── Light cleaning for tonight's operations ────────────────────────────────
fish_streams <- streams |> filter(FTYPE == "Stream/River", STREAM_ORDER >= 2)
parcels <- parcels |>
  mutate(parcel_key = coalesce(PARCEL_ID, paste0("NOID-", row_number() - 1)),
         area_acres = round(as.numeric(st_area(parcels)) / SQFT_PER_ACRE, 3))  # from geometry, not AREA_SQFT
st_agr(parcels) <- "constant"   # new columns describe whole parcels too
issaquah_ws <- watersheds |> filter(WATERSHED_NAME == "Issaquah Creek")

for (nm in c("watersheds", "streams", "wria8", "parcels", "salmon_obs", "fish_streams")) {
  x <- get(nm)
  cat(sprintf("  %-13s %4d features | %-16s | EPSG:%s\n", nm, nrow(x),
              as.character(st_geometry_type(x)[1]), st_crs(x)$epsg))
}

#### ESRI first: the same geodatabases in ArcPy

In ArcGIS Pro you'd point the geoprocessing environment at the File GDB and write results to a scratch geodatabase. The cell below sets that up for every ArcPy cell in this notebook: `arcpy.env.workspace` is the Module 3 GDB, and outputs go to `outputs/module4.gdb`.

🟢 **Python**

In [ ]:
# [Python / ArcPy] Section 1 — Geoprocessing environment for the ArcPy cells
# Runs in ArcGIS Pro's Notebook or a cloned arcgispro-py3 env; skips gracefully elsewhere.
import os
DATA = Path('your/data/path/on/your/machine')
try:
    import arcpy
    WS_FGDB     = str(WS_GDB.resolve())
    PARCEL_FGDB = str(PARCEL_GDB.resolve())
    OUT_GDB     = str((OUT / 'module4.gdb').resolve())
    if not arcpy.Exists(OUT_GDB):
        arcpy.management.CreateFileGDB(str(OUT.resolve()), 'module4.gdb')
    arcpy.env.workspace = WS_FGDB
    arcpy.env.overwriteOutput = True
    arcpy.env.outputCoordinateSystem = arcpy.SpatialReference(COURSE_EPSG)

    print("Feature classes:", arcpy.ListFeatureClasses())
    sr = arcpy.Describe('streams').spatialReference
    print(f"streams CRS: {sr.name} | WKID {sr.factoryCode} | linear unit: {sr.linearUnitName}")   # Foot_US
    print("ArcPy outputs will go to:", OUT_GDB)
    HAVE_ARCPY = True
except ModuleNotFoundError:
    HAVE_ARCPY = False
    print("arcpy isn't available in this kernel — ArcPy cells will skip. The GeoPandas and sf cells do the same work.")

---
## Section 2 — Buffers: Creating Distance Zones

A **buffer** creates a new polygon layer by expanding each feature outward by a fixed distance. The result is a proximity zone you can use for:
- Riparian setbacks (e.g., land within 150 ft of a fish-bearing stream — tonight's example)
- Impact assessments (e.g., parcels within 1,000 ft of a proposed pipeline)
- Service-area screening (e.g., water-quality sites within ½ mile of a salmon redd)

**Remember:** buffer in a **projected CRS**, and know its **units**. A buffer of 150 on a layer in EPSG:4326 means 150 *degrees*. In EPSG:2926 it means 150 **US survey feet**, and in UTM (EPSG:32610) it means 150 **metres**. Same number, three very different zones.

| Operation | ArcPy | Python | R |
|---|---|---|---|
| Buffer | `arcpy.analysis.Buffer(in, out, "150 Feet")` | `gdf.buffer(150)` | `st_buffer(x, dist = 150)` |
| Buffer + dissolve | `Buffer(..., dissolve_option="ALL")` | `gdf.buffer(150)` → `.dissolve()` | `st_buffer()` → `st_union()` |

> **Where does 150 ft come from?** It's a typical value, in the range used in Puget Sound for fish-bearing streams. Real buffer widths depend on the jurisdiction and stream type.

> **ArcGIS Pro equivalent:** Geoprocessing → Analysis Tools → Proximity → Buffer (or Pairwise Buffer). The logic is identical — code just makes it reproducible.

### 2a — Buffer with ArcPy
This section you need to run locally where you have access to the `arcpy` package

🟢 **Python**

In [ ]:
# [Python / ArcPy] Buffer fish-bearing streams by 150 ft
try:
    import arcpy
HAVE_ARCPY = True
except ModuleNotFoundError:
    HAVE_ARCPY = False
    print("arcpy isn't available in this kernel — ArcPy cells will skip. The GeoPandas and sf cells do the same work.")
if HAVE_ARCPY:
    # Select fish-bearing streams with an attribute query (same filter as the setup cell)
    fish_lyr = arcpy.management.MakeFeatureLayer(
        os.path.join(WS_FGDB, 'streams'), 'fish_streams_lyr',
        where_clause="FTYPE = 'Stream/River' AND STREAM_ORDER >= 2")
    print("Fish-bearing stream segments:", arcpy.management.GetCount(fish_lyr)[0])

    # One buffer polygon per stream segment (keeps stream attributes)
    buf_fc = os.path.join(OUT_GDB, f'fish_streams_buf{BUFFER_FT}ft')
    arcpy.analysis.Buffer(fish_lyr, buf_fc, f"{BUFFER_FT} Feet")

    # Dissolved into a single riparian zone — what you'd overlay parcels against
    zone_fc = os.path.join(OUT_GDB, f'riparian_zone_{BUFFER_FT}ft')
    arcpy.analysis.PairwiseBuffer(fish_lyr, zone_fc, f"{BUFFER_FT} Feet", dissolve_option="ALL")

    print("Buffer features:", arcpy.management.GetCount(buf_fc)[0], "| dissolved:", arcpy.management.GetCount(zone_fc)[0])
    with arcpy.da.SearchCursor(zone_fc, ["SHAPE@AREA"]) as cur:
        print(f"Riparian zone area: {next(cur)[0] / SQFT_PER_ACRE:,.0f} acres")
else:
    print("Skipped — no arcpy. Section 2b does the same buffer in GeoPandas.")

### 2b — Buffer in Python with `geopandas`

In [ ]:
# [Python] Buffer fish-bearing streams by 150 ft and verify the result
# ── Step 1: Confirm CRS and units before touching geometry ─────────────────
print(f"CRS: {fish_streams.crs.name}")
print(f"Units: {fish_streams.crs.axis_info[0].unit_name}")       # US survey foot
assert fish_streams.crs.is_projected, "Reproject to a projected CRS before buffering!"

# ── Step 2: Buffer — .buffer() returns a GeoSeries (geometry only) ─────────
buf = fish_streams.buffer(BUFFER_FT)
print(f"\n.buffer() returns: {type(buf).__name__}")

# ── Step 3: Wrap in a GeoDataFrame to keep the stream attributes ──────────
stream_buf = gpd.GeoDataFrame(
    fish_streams[['STREAM_ID', 'STREAM_NAME', 'WATERSHED_ID', 'STREAM_ORDER']].copy(),
    geometry=buf, crs=fish_streams.crs)
print(f"Buffer features: {len(stream_buf)} — matches input ({len(fish_streams)})")

# ── Step 4: Dissolve to one riparian zone (overlapping buffers merged) ─────
riparian_zone = stream_buf.dissolve()[['geometry']]
riparian_zone['buffer_ft'] = BUFFER_FT
sum_acres = stream_buf.area.sum() / SQFT_PER_ACRE
zone_acres = riparian_zone.area.iloc[0] / SQFT_PER_ACRE
print(f"Sum of individual buffers: {sum_acres:,.0f} ac | dissolved zone: {zone_acres:,.0f} ac "
      f"(overlap removed: {sum_acres - zone_acres:,.0f} ac)")

# ── Step 5: Plot to verify — zoom to the Issaquah parcels ─────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 6))
watersheds.plot(ax=axes[0], color='#f2f2f2', edgecolor='#999999', linewidth=0.6)
riparian_zone.plot(ax=axes[0], color='#9ecae1', edgecolor='none')
fish_streams.plot(ax=axes[0], color='#08519c', linewidth=0.6)
axes[0].set_title('WRIA 8 — fish-bearing streams, 150 ft buffer')

xmin, ymin, xmax, ymax = parcels.total_bounds
riparian_zone.plot(ax=axes[1], color='#9ecae1', edgecolor='#3182bd', linewidth=0.5)
parcels.plot(ax=axes[1], facecolor='none', edgecolor='#7f7f7f', linewidth=0.4)
fish_streams.plot(ax=axes[1], color='#08519c', linewidth=1.2)
axes[1].set_xlim(xmin, xmax); axes[1].set_ylim(ymin, ymax)
axes[1].set_title('Zoom: Issaquah public parcels (grey) vs. buffer (blue)')
for ax in axes: ax.set_axis_off()
plt.suptitle(f'Buffer: {BUFFER_FT} ft around fish-bearing streams (EPSG:{COURSE_EPSG}, US ft)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(OUT / 'm4_buffer_streams.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# [Python] Export both buffer layers to one GeoPackage, then round-trip check
buf_gpkg = OUT / 'module4_buffers.gpkg'
stream_buf.to_file(buf_gpkg, layer=f'stream_buffers_{BUFFER_FT}ft', driver='GPKG')
riparian_zone.to_file(buf_gpkg, layer=f'riparian_zone_{BUFFER_FT}ft', driver='GPKG')
print(f"Saved: {buf_gpkg}  →  layers {list_layers(buf_gpkg)}")

check = gpd.read_file(buf_gpkg, layer=f'stream_buffers_{BUFFER_FT}ft')
print(f"Re-loaded: {len(check)} features | EPSG:{check.crs.to_epsg()} | columns: {list(check.columns)}")

### 2c — Buffer in R with `sf`

🔵 **`R`**

In [ ]:
# [R] Buffer fish-bearing streams by 150 ft and verify the result
# ── Confirm CRS and units first ───────────────────────────────────────────
cat("CRS:", st_crs(fish_streams)$Name, "| units:", st_crs(fish_streams)$units, "\n")   # us-ft
stopifnot(!st_is_longlat(fish_streams))

# ── Buffer: st_buffer() returns a full sf object, attributes included ──────
stream_buf <- st_buffer(fish_streams, dist = BUFFER_FT)
cat("Buffer features:", nrow(stream_buf), "— matches input", nrow(fish_streams), "\n")

# ── Dissolve to one riparian zone ──────────────────────────────────────────
riparian_zone <- st_sf(buffer_ft = BUFFER_FT, geometry = st_union(stream_buf), agr = "constant")
cat(sprintf("Dissolved riparian zone: %s acres\n",
            format(round(as.numeric(st_area(riparian_zone)) / SQFT_PER_ACRE), big.mark = ",")))

# ── Plot to verify — zoom to the Issaquah parcels ──────────────────────────
bb <- st_bbox(parcels)
plot(st_geometry(riparian_zone), col = "#9ecae1", border = "#3182bd",
     xlim = bb[c("xmin", "xmax")], ylim = bb[c("ymin", "ymax")],
     main = "150 ft riparian buffer vs. Issaquah public parcels (EPSG:2926)")
plot(st_geometry(parcels), border = "#7f7f7f", lwd = 0.4, add = TRUE)
plot(st_geometry(fish_streams), col = "#08519c", lwd = 1.2, add = TRUE)

In [ ]:
# [R] Export both buffer layers to one GeoPackage, then round-trip check
buf_gpkg <- file.path(OUT, "module4_buffers_r.gpkg")
st_write(stream_buf,    buf_gpkg, layer = paste0("stream_buffers_", BUFFER_FT, "ft"), delete_dsn = TRUE, quiet = TRUE)
st_write(riparian_zone, buf_gpkg, layer = paste0("riparian_zone_",  BUFFER_FT, "ft"), quiet = TRUE)
print(st_layers(buf_gpkg))

check <- st_read(buf_gpkg, layer = paste0("stream_buffers_", BUFFER_FT, "ft"), quiet = TRUE)
cat("Re-loaded:", nrow(check), "features | EPSG:", st_crs(check)$epsg, "\n")

### 🔧 Try it yourself — Buffers

1. Change `BUFFER_FT` to 100 and then 300 and re-run 2b. How does the dissolved riparian area change? Is it linear in the distance? Why not?
2. Buffer `salmon_obs` (points) by ½ mile (2,640 ft). What does the output look like? How many observation buffers overlap?
3. Run `fish_streams.to_crs(4326).buffer(150)` and look at the result (and the warning). How large is that "buffer"?
4. **Units trap:** a colleague's script says `buffer(50)` and the comment says "50 m setback". In EPSG:2926 what distance did they actually buffer? What number should they have used?
5. **ArcPy connection:** in `arcpy.analysis.Buffer()`, which parameter maps to `dist=` in `st_buffer()`? What does `dissolve_option="ALL"` correspond to in GeoPandas and in sf?

---
## Section 3 — Clip & Overlay: Cutting and Combining Layers

Clip and overlay operations use one layer's geometry to modify another. With the Module 3 data they answer questions like:
- *Which stream segments fall inside the Issaquah Creek watershed?* → **Clip**
- *How much of each public parcel lies inside the riparian buffer — and keep both layers' attributes?* → **Intersection**
- *What's left of each parcel outside the buffer (the unconstrained area)?* → **Difference**
- *Give me parcels and buffer merged, with every piece labelled by what it overlaps.* → **Union**

| Operation | Keeps | ArcPy | Python | R |
|---|---|---|---|---|
| Clip | A inside B; A's attributes only | `arcpy.analysis.Clip(A, B, out)` | `gpd.clip(A, B)` | `st_intersection(A, st_union(B))` |
| Intersection | Overlap; both A and B attributes | `arcpy.analysis.Intersect([A, B], out)` | `gpd.overlay(A, B, how='intersection')` | `st_intersection(A, B)` |
| Difference | A minus B overlap | `arcpy.analysis.Erase(A, B, out)` *(or Pairwise Erase)* | `gpd.overlay(A, B, how='difference')` | `st_difference(A, st_union(B))` |
| Union | All pieces of A and B | `arcpy.analysis.Union([A, B], out)` | `gpd.overlay(A, B, how='union')` | *(no one-call equivalent — see 3c)* |

> **Clip vs. Intersection:** `gpd.clip()` is faster and keeps only A's columns. Use clip to trim geometry; use `overlay(how='intersection')` when you need columns from *both* layers.

> **R gotcha:** `st_union(A, B)` is **not** the Union tool. With two arguments it dissolves geometry pairwise. For clip and difference in R, dissolve the mask first with `st_union(B)` so each A feature is cut once rather than once per B feature.

### 3a — Clip & Overlay with ArcPy

🟢 **Python**

In [ ]:
# [Python / ArcPy] Clip, Intersect, Erase, Union — streams, watersheds, parcels, riparian zone
if HAVE_ARCPY:
    arcpy.env.workspace = OUT_GDB
    ws_fc      = os.path.join(WS_FGDB, 'watersheds')
    streams_fc = os.path.join(WS_FGDB, 'streams')
    parcels_fc = os.path.join(PARCEL_FGDB, 'Parcels')
    zone_fc    = os.path.join(OUT_GDB, f'riparian_zone_{BUFFER_FT}ft')   # from 2a

    # Clip: streams inside the Issaquah Creek watershed
    issaquah_lyr = arcpy.management.MakeFeatureLayer(ws_fc, 'issaquah_lyr', "WATERSHED_NAME = 'Issaquah Creek'")
    arcpy.analysis.Clip(streams_fc, issaquah_lyr, 'streams_issaquah')

    # Intersection: parcel pieces inside the riparian zone, attributes from both
    arcpy.analysis.Intersect([parcels_fc, zone_fc], 'parcels_in_riparian')

    # Difference: parcel area outside the riparian zone
    arcpy.analysis.Erase(parcels_fc, zone_fc, 'parcels_outside_riparian')

    # Union: every piece of both layers
    arcpy.analysis.Union([parcels_fc, zone_fc], 'parcels_union_riparian')

    for fc in ['streams_issaquah', 'parcels_in_riparian', 'parcels_outside_riparian', 'parcels_union_riparian']:
        print(f"  {fc:<26} {arcpy.management.GetCount(fc)[0]:>5} features")
else:
    print("Skipped — no arcpy. Section 3b runs the same four operations in GeoPandas.")

### 3b — Clip & Overlay in Python with `geopandas`

In [ ]:
# [Python] Clip and overlay operations with geopandas
# ── CRS check — always before any multi-layer operation ───────────────────
for name, gdf in {'streams': streams, 'issaquah_ws': issaquah_ws,
                  'parcels': parcels, 'riparian_zone': riparian_zone}.items():
    print(f"  {name:<14} EPSG:{gdf.crs.to_epsg()}")
assert streams.crs == issaquah_ws.crs == parcels.crs == riparian_zone.crs, "CRS mismatch — reproject first!"
print("CRS match ✓\n")

# ── Clip: streams inside the Issaquah Creek watershed (streams' columns only) ─
streams_issaquah = gpd.clip(streams, issaquah_ws)
print(f"Clip — streams: {len(streams)} in WRIA 8 → {len(streams_issaquah)} in Issaquah Creek")
print(f"       stream miles: {streams_issaquah.length.sum() / 5280:,.1f}")
print(f"       columns: {list(streams_issaquah.columns)}")

# ── Intersection: parcel pieces inside the riparian zone, columns from both ─
parcels_in_rip = gpd.overlay(parcels, riparian_zone, how='intersection', keep_geom_type=True)
parcels_in_rip['riparian_acres'] = parcels_in_rip.area / SQFT_PER_ACRE
print(f"\nIntersection — {len(parcels_in_rip)} of {len(parcels)} parcels touch the {BUFFER_FT} ft zone "
      f"({parcels_in_rip['riparian_acres'].sum():,.1f} riparian acres)")
print(f"       new column from the buffer layer: 'buffer_ft' → {'buffer_ft' in parcels_in_rip.columns}")

# ── Difference: what's left of each parcel outside the zone ───────────────
parcels_outside = gpd.overlay(parcels, riparian_zone, how='difference', keep_geom_type=True)
print(f"\nDifference — {len(parcels_outside)} parcels keep area outside the zone "
      f"({parcels_outside.area.sum() / SQFT_PER_ACRE:,.0f} ac)")

# ── Union: every piece of both layers, each labelled by what it overlaps ──
# (clip the zone to the parcel extent first so the union stays local and fast)
from shapely.geometry import box
zone_local = gpd.clip(riparian_zone, box(*parcels.total_bounds))
parcels_union = gpd.overlay(parcels, zone_local, how='union', keep_geom_type=True)
print(f"\nUnion — {len(parcels_union)} pieces "
      f"(parcel only: {(parcels_union['buffer_ft'].isna()).sum()}, "
      f"zone only: {(parcels_union['parcel_key'].isna()).sum()}, "
      f"both: {(parcels_union['parcel_key'].notna() & parcels_union['buffer_ft'].notna()).sum()})")

# Quick check: inside + outside should add back up to the original parcel area
total = parcels.area.sum(); split = parcels_in_rip.area.sum() + parcels_outside.area.sum()
print(f"\nArea check: original {total/SQFT_PER_ACRE:,.1f} ac vs. inside+outside {split/SQFT_PER_ACRE:,.1f} ac")

In [ ]:
# [Python] Plot all four operations side by side
xmin, ymin, xmax, ymax = parcels.total_bounds
fig, axes = plt.subplots(2, 2, figsize=(14, 11))

# Clip
issaquah_ws.plot(ax=axes[0, 0], color='#f2f2f2', edgecolor='#cc0000', linewidth=1.2)
streams.plot(ax=axes[0, 0], color='#cfcfcf', linewidth=0.4)
streams_issaquah.plot(ax=axes[0, 0], color='#08519c', linewidth=0.9)
b = issaquah_ws.total_bounds
axes[0, 0].set_xlim(b[0] - 5000, b[2] + 5000); axes[0, 0].set_ylim(b[1] - 5000, b[3] + 5000)
axes[0, 0].set_title('Clip — streams inside Issaquah Creek watershed (red)')

# Intersection
parcels.plot(ax=axes[0, 1], facecolor='none', edgecolor='#bdbdbd', linewidth=0.3)
parcels_in_rip.plot(ax=axes[0, 1], color='#08519c', edgecolor='#08519c', linewidth=0.6)
axes[0, 1].set_title('Intersection — parcel area inside the 150 ft zone')

# Difference
parcels_outside.plot(ax=axes[1, 0], color='#c7e9c0', edgecolor='#41ab5d', linewidth=0.3)
riparian_zone.boundary.plot(ax=axes[1, 0], color='#3182bd', linewidth=0.6)
axes[1, 0].set_title('Difference — parcel area outside the zone (holes = buffer)')

# Union
labels = pd.Series('parcel only', index=parcels_union.index)
labels[parcels_union['parcel_key'].isna()] = 'buffer only'
labels[parcels_union['parcel_key'].notna() & parcels_union['buffer_ft'].notna()] = 'both'
parcels_union.assign(piece=labels).plot(ax=axes[1, 1], column='piece', categorical=True, legend=True,
                                        cmap='Set2', edgecolor='white', linewidth=0.2)
axes[1, 1].set_title('Union — every piece of both layers')

for ax in [axes[0, 1], axes[1, 0], axes[1, 1]]:
    ax.set_xlim(xmin, xmax); ax.set_ylim(ymin, ymax)
for ax in axes.flat: ax.set_axis_off()
plt.suptitle('Clip & Overlay — geopandas on the Module 3 data', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(OUT / 'm4_overlay_operations.png', dpi=150, bbox_inches='tight')
plt.show()

### 3c — Clip & Overlay in R with `sf`

🔵 **`R`**

In [ ]:
# [R] Clip and overlay operations with sf
stopifnot(st_crs(streams) == st_crs(issaquah_ws), st_crs(parcels) == st_crs(riparian_zone))

# ── Clip: streams inside Issaquah Creek (mask has one feature, so no dissolve needed)
streams_issaquah <- st_intersection(streams, st_geometry(issaquah_ws))
cat("Clip — streams:", nrow(streams), "→", nrow(streams_issaquah), "in Issaquah Creek |",
    round(sum(as.numeric(st_length(streams_issaquah))) / 5280, 1), "stream miles\n")

# ── Intersection: parcel pieces inside the riparian zone, attributes from both
parcels_in_rip <- st_intersection(parcels, riparian_zone) |>
  mutate(riparian_acres = as.numeric(st_area(geometry)) / SQFT_PER_ACRE)
cat("Intersection —", nrow(parcels_in_rip), "parcels touch the zone |",
    round(sum(parcels_in_rip$riparian_acres), 1), "riparian acres | has buffer_ft:",
    "buffer_ft" %in% names(parcels_in_rip), "\n")

# ── Difference: parcel area outside the zone (dissolved mask → one cut per parcel)
parcels_outside <- st_difference(parcels, st_union(riparian_zone))
cat("Difference —", nrow(parcels_outside), "parcels keep area outside the zone\n")

# ── Union: sf has no single Union-tool call; build it from the three pieces
zone_only <- st_difference(st_intersection(riparian_zone, st_as_sfc(st_bbox(parcels))), st_union(parcels))
parcels_union <- bind_rows(
  mutate(parcels_in_rip,  piece = "both"),
  mutate(parcels_outside, piece = "parcel only"),
  mutate(zone_only,       piece = "buffer only"))
print(table(parcels_union$piece))

In [ ]:
# [R] Plot the clip and the intersection to verify the geometry
op <- par(mfrow = c(1, 2), mar = c(1, 1, 3, 1))
plot(st_geometry(issaquah_ws), border = "#cc0000", lwd = 1.5, main = "Clip — streams in Issaquah Creek")
plot(st_geometry(streams_issaquah), col = "#08519c", add = TRUE)

bb <- st_bbox(parcels)
plot(st_geometry(parcels), border = "#bdbdbd", lwd = 0.3,
     xlim = bb[c("xmin", "xmax")], ylim = bb[c("ymin", "ymax")], main = "Intersection — parcels ∩ 150 ft zone")
plot(st_geometry(parcels_in_rip), col = "#3182bd", border = NA, add = TRUE)
par(op)

### 🔧 Try it yourself — Clip & Overlay

1. Swap the layer order: `gpd.overlay(riparian_zone, parcels, how='difference')`. What do you get now, and when would you want it?
2. Compare the columns of `gpd.clip(parcels, riparian_zone)` and `gpd.overlay(parcels, riparian_zone, how='intersection')`. Which one has `buffer_ft`? Why?
3. Clip `fish_streams` to `issaquah_ws` and compute stream miles by `STREAM_ORDER`. Which order contributes the most miles?
4. Which **five parcels** have the most riparian acres? Which owner types are they? (*Hint:* sort `parcels_in_rip` by `riparian_acres`.)
5. **ArcPy connection:** you need the area of each parcel inside the buffer *and* the parcel's `OWNER_TYPE`. Would you use the Clip tool or the Intersect tool in Pro? Which one did we use in 2b?

---
## Section 4 — Spatial Joins: Location as the Join Key

A **spatial join** transfers attributes from one layer to another based on where they sit, with no shared ID column needed. In Module 3's Challenge A you previewed this: *which watershed is each salmon observation in?* Tonight we do it properly, plus a nearest-feature join.

**Common predicates:**

| Predicate | Meaning | Example with our data |
|---|---|---|
| `within` | Left feature is completely inside a right feature | Salmon observation inside a watershed |
| `intersects` | Any part of the geometries touch or overlap | Parcels touching a watershed (some touch two!) |
| `contains` | Left feature fully encloses right feature | Watersheds that contain observations |
| nearest | Closest feature, even with no overlap | Nearest water-temperature site to each observation |

| Operation | ArcPy | Python | R |
|---|---|---|---|
| Spatial join | `arcpy.analysis.SpatialJoin(target, join, out, match_option="WITHIN")` | `gpd.sjoin(left, right, how, predicate)` | `st_join(x, y, join = st_within)` |
| Nearest join | `SpatialJoin(..., match_option="CLOSEST", search_radius="2640 Feet")` | `gpd.sjoin_nearest(left, right, max_distance=, distance_col=)` | `st_join(x, y, join = st_nearest_feature)` |

> **Watch for row multiplication:** if a feature matches more than one right-hand feature, you get one row per match. Check `len(joined)` against `len(left)` after **every** join.

### 4a — Spatial Join with ArcPy

🟢 **Python**

In [ ]:
# [Python / ArcPy] Spatial join: watershed attributes → salmon observations; parcels → watersheds
if HAVE_ARCPY:
    arcpy.env.workspace = OUT_GDB
    ws_fc      = os.path.join(WS_FGDB, 'watersheds')
    parcels_fc = os.path.join(PARCEL_FGDB, 'Parcels')
    # Use the live hosted layer if you're signed in to Pro; otherwise the offline backup GeoPackage
    # (ArcGIS Pro addresses GeoPackage layers as <file.gpkg>\main.<layer>)
    obs_in = os.path.join(str((DATA / 'salmon_observations_hosted_backup.gpkg').resolve()), 'main.salmon_observations')
    # obs_in = SALMON_OBS_URL

    # Points in polygons — JOIN_ONE_TO_ONE keeps one row per observation (like how='left')
    arcpy.analysis.SpatialJoin(obs_in, ws_fc, 'salmon_obs_ws',
                               join_operation="JOIN_ONE_TO_ONE", join_type="KEEP_ALL",
                               match_option="WITHIN")
    print("Observations in:", arcpy.management.GetCount(obs_in)[0],
          "| joined out:", arcpy.management.GetCount('salmon_obs_ws')[0])

    # Polygons to polygons — HAVE_THEIR_CENTER_IN gives each parcel exactly one watershed
    arcpy.analysis.SpatialJoin(parcels_fc, ws_fc, 'parcels_ws_onetomany',
                               join_operation="JOIN_ONE_TO_MANY", match_option="INTERSECT")
    arcpy.analysis.SpatialJoin(parcels_fc, ws_fc, 'parcels_ws_center',
                               join_operation="JOIN_ONE_TO_ONE", match_option="HAVE_THEIR_CENTER_IN")
    print("Parcels:", arcpy.management.GetCount(parcels_fc)[0],
          "| ONE_TO_MANY + INTERSECT:", arcpy.management.GetCount('parcels_ws_onetomany')[0],
          "| ONE_TO_ONE + HAVE_THEIR_CENTER_IN:", arcpy.management.GetCount('parcels_ws_center')[0])
else:
    print("Skipped — no arcpy. Section 4b runs the same joins in GeoPandas.")

### 4b — Spatial Join in Python with `geopandas`

In [ ]:
# [Python] Spatial join: which watershed is each salmon observation in?
assert salmon_obs.crs == watersheds.crs, "CRS mismatch — reproject first!"

ws_cols = watersheds[['WATERSHED_ID', 'WATERSHED_NAME', 'HUC12', 'geometry']]
obs_ws = gpd.sjoin(
    salmon_obs,          # left: points — get attributes added
    ws_cols,             # right: polygons — provide attributes
    how='left',          # keep every observation, matched or not
    predicate='within')  # point must be inside the polygon

# ── Verify row counts ──────────────────────────────────────────────────────
print(f"Observations before join: {len(salmon_obs)} | after: {len(obs_ws)}")
print("Row counts match ✓" if len(obs_ws) == len(salmon_obs) else "⚠  Row count changed — check for multiple matches")
print(f"Unmatched (outside every watershed): {obs_ws['WATERSHED_ID'].isna().sum()}")
print(f"New columns: {[c for c in obs_ws.columns if c not in salmon_obs.columns]}")

# ── Summarise: observations and fish counted per watershed, by species ────
species_by_ws = pd.crosstab(obs_ws['WATERSHED_NAME'], obs_ws['species'], margins=True, margins_name='Total')
print("\nObservations per watershed × species:")
print(species_by_ws.to_string())

fish_by_ws = obs_ws.groupby('WATERSHED_NAME').agg(n_obs=('obs_id', 'count'), fish_counted=('count', 'sum'))
print("\nFish counted per watershed:")
print(fish_by_ws.sort_values('fish_counted', ascending=False).to_string())

In [ ]:
# [Python] Nearest join: attach the closest water-temperature site to each observation
# monitoring_sites.csv → points, exactly the Module 3 tabular → spatial pattern
ms = pd.read_csv(DATA / 'monitoring_sites.csv')
sites = gpd.GeoDataFrame(ms, geometry=gpd.points_from_xy(ms['longitude'], ms['latitude']),  # x = lon!
                         crs=4326).to_crs(COURSE_EPSG)
print(f"Water-quality sites: {len(sites)} | EPSG:{sites.crs.to_epsg()}")

obs_temp = gpd.sjoin_nearest(
    salmon_obs[['obs_id', 'species', 'count', 'stream_name', 'geometry']],
    sites[['site_id', 'value', 'sample_date', 'geometry']].rename(columns={'value': 'temp_c'}),
    how='left',
    max_distance=2640,        # ½ mile, in feet — no match beyond this
    distance_col='dist_ft')

print(f"Observations: {len(salmon_obs)} → joined rows: {len(obs_temp)} "
      f"(ties at equal distance would add rows)")
print(f"No site within ½ mile: {obs_temp['site_id'].isna().sum()}")
print(f"Median distance to nearest site: {obs_temp['dist_ft'].median():,.0f} ft")
print("\nMean nearby water temperature (°C) by species:")
print(obs_temp.groupby('species')['temp_c'].agg(['count', 'mean']).round(1).to_string())

In [ ]:
# [Python] Map the join result: observation counts back onto the watershed polygons
ws_counts = watersheds.merge(
    obs_ws.groupby('WATERSHED_ID').size().rename('n_obs').reset_index(),
    on='WATERSHED_ID', how='left').fillna({'n_obs': 0})

fig, ax = plt.subplots(figsize=(9, 10))
ws_counts.plot(ax=ax, column='n_obs', cmap='Blues', edgecolor='#636363', linewidth=0.5,
               legend=True, legend_kwds={'label': 'Salmon observations', 'shrink': 0.6})
fish_streams.plot(ax=ax, color='#6baed6', linewidth=0.4)
markers = {'Chinook': 'o', 'Coho': 's', 'Sockeye': '^'}
for sp, grp in obs_ws.groupby('species'):
    grp.plot(ax=ax, marker=markers.get(sp, 'o'), color='#d94801', edgecolor='black',
             markersize=28, linewidth=0.4, label=sp)
ax.legend(title='Species', loc='lower left')
ax.set_title('Salmon observations per watershed (via spatial join)', fontsize=12, fontweight='bold')
ax.set_axis_off()
plt.tight_layout()
plt.savefig(OUT / 'm4_spatial_join_watersheds.png', dpi=150, bbox_inches='tight')
plt.show()

### 4c — Spatial Join in R with `sf`

🔵 **`R`**

In [ ]:
# [R] Spatial join: watershed attributes → salmon observations
stopifnot(st_crs(salmon_obs) == st_crs(watersheds))

obs_ws <- st_join(
  salmon_obs,                                               # left: points — get attributes
  watersheds |> select(WATERSHED_ID, WATERSHED_NAME, HUC12),  # right: polygons
  join = st_within,                                         # predicate
  left = TRUE)                                              # keep all observations

cat("Observations before:", nrow(salmon_obs), "| after:", nrow(obs_ws),
    "| unmatched:", sum(is.na(obs_ws$WATERSHED_ID)), "\n\n")

# Observations per watershed × species
print(addmargins(table(obs_ws$WATERSHED_NAME, obs_ws$species)))

obs_ws |>
  st_drop_geometry() |>
  group_by(WATERSHED_NAME) |>
  summarise(n_obs = n(), fish_counted = sum(count)) |>
  arrange(desc(fish_counted)) |>
  print()

In [ ]:
# [R] Nearest join: closest water-temperature site to each observation
ms <- read.csv(file.path(DATA, "monitoring_sites.csv"))
sites <- st_as_sf(ms, coords = c("longitude", "latitude"), crs = 4326) |>   # x = longitude first
  st_transform(COURSE_EPSG) |>
  select(site_id, temp_c = value, sample_date)

obs_temp <- st_join(salmon_obs |> select(obs_id, species, count), sites, join = st_nearest_feature)
nearest_idx <- st_nearest_feature(salmon_obs, sites)
obs_temp$dist_ft <- as.numeric(st_distance(salmon_obs, sites[nearest_idx, ], by_element = TRUE))

# st_nearest_feature has no max distance — apply the ½ mile rule ourselves
obs_temp <- obs_temp |> mutate(temp_c = if_else(dist_ft <= 2640, temp_c, NA_real_))
cat("No site within ½ mile:", sum(is.na(obs_temp$temp_c)),
    "| median distance:", round(median(obs_temp$dist_ft)), "ft\n")
obs_temp |> st_drop_geometry() |> group_by(species) |>
  summarise(n_with_site = sum(!is.na(temp_c)), mean_temp_c = round(mean(temp_c, na.rm = TRUE), 1)) |>
  print()

### 🔧 Try it yourself — Spatial Joins

1. Change `predicate='within'` to `predicate='intersects'` for the observations. Does the row count change? Now try the same switch with **parcels** → watersheds. Why do the two layers behave differently?
2. Join the **partner agency** points (`salmon_obs_partner_agency.geojson`, reproject first) to watersheds with `how='left'` and then `how='inner'`. How many rows are dropped, and where are those points?
3. Re-run the nearest join with `max_distance=1320` (¼ mile). How many observations lose their temperature site?
4. Using `obs_ws`, which watershed has the most **Chinook** fish counted (`count` column, not the number of observations)?
5. **ArcPy connection:** `JOIN_ONE_TO_ONE` vs. `JOIN_ONE_TO_MANY` — which one behaves like `gpd.sjoin()` by default? What do you add in GeoPandas to get one-to-one behaviour?

---
## Section 5 — Complete Workflow: Chaining All Three Operations

In real GIS work, buffer, clip and spatial join are almost never used alone. The real power is in **chaining** them:

```
Load (GDBs) → CRS check → Spatial Join → Select → Clip → Buffer → Overlay → Summarise → Export
```

> **Scenario (tonight's exercise):** For the watersheds where **Chinook** were observed, how many acres of publicly owned parcels lie within **150 ft** of a fish-bearing stream — by watershed and owner type?

The chain:
1. **Spatial join** salmon observations → watersheds; keep watersheds with ≥ 1 Chinook observation
2. **Clip** fish-bearing streams to those watersheds
3. **Buffer** the clipped streams by 150 ft and dissolve
4. **Overlay (intersection)** parcels × buffer × watersheds — riparian parcel area, labelled with owner type *and* watershed
5. **Summarise** riparian acres by watershed and owner type
6. **Export** every intermediate layer to one GeoPackage, plus a CSV and a map

**ArcPy equivalents are shown inline** so you can see the same logic across both surfaces. Change `BUFFER_FT` or `TARGET_SPECIES` at the top of the cell and re-run the whole analysis in seconds. That's the automation payoff.

### 5a — Complete workflow in Python

🟢 **Python**

In [ ]:
# [Python] Complete chained workflow: Spatial Join → Clip → Buffer → Overlay → Summarise → Export
TARGET_SPECIES = 'Chinook'
WORKFLOW_BUFFER_FT = 150

# ═════════════════════════════════════════════════════════════
# Step 1: Inputs (already read from the GDBs in Section 1) + CRS check
# ArcPy: arcpy.env.outputCoordinateSystem = arcpy.SpatialReference(2926)
# ═════════════════════════════════════════════════════════════
inputs = {'watersheds': watersheds, 'fish_streams': fish_streams, 'parcels': parcels, 'salmon_obs': salmon_obs}
assert len({g.crs.to_epsg() for g in inputs.values()}) == 1, "CRS mismatch — reproject first!"
print(f"Step 1: {', '.join(f'{k}={len(v)}' for k, v in inputs.items())} | all EPSG:{COURSE_EPSG}")

# ═════════════════════════════════════════════════════════════
# Step 2: Spatial join — which watersheds have the target species?
# ArcPy: arcpy.analysis.SpatialJoin(obs, watersheds, obs_ws, match_option="WITHIN")
# ═════════════════════════════════════════════════════════════
obs_ws_wf = gpd.sjoin(salmon_obs, watersheds[['WATERSHED_ID', 'WATERSHED_NAME', 'geometry']],
                      how='inner', predicate='within')
target_ids = obs_ws_wf.loc[obs_ws_wf['species'] == TARGET_SPECIES, 'WATERSHED_ID'].unique()
target_ws = watersheds[watersheds['WATERSHED_ID'].isin(target_ids)].copy()
print(f"\nStep 2: {TARGET_SPECIES} observed in {len(target_ws)} of {len(watersheds)} watersheds:")
print("        " + ", ".join(sorted(target_ws['WATERSHED_NAME'])))

# ═════════════════════════════════════════════════════════════
# Step 3: Clip fish-bearing streams to those watersheds
# ArcPy: arcpy.analysis.Clip(fish_streams, target_ws, streams_clip)
# ═════════════════════════════════════════════════════════════
streams_clip = gpd.clip(fish_streams, target_ws)
print(f"\nStep 3: Clipped streams — {len(fish_streams)} → {len(streams_clip)} segments "
      f"({streams_clip.length.sum() / 5280:,.1f} mi)")

# ═════════════════════════════════════════════════════════════
# Step 4: Buffer 150 ft and dissolve to one zone
# ArcPy: arcpy.analysis.PairwiseBuffer(streams_clip, rip_buf, "150 Feet", dissolve_option="ALL")
# ═════════════════════════════════════════════════════════════
rip_buf = gpd.GeoDataFrame({'buffer_ft': [WORKFLOW_BUFFER_FT]},
                           geometry=[streams_clip.buffer(WORKFLOW_BUFFER_FT).union_all()
                                     if hasattr(streams_clip, 'union_all')
                                     else streams_clip.buffer(WORKFLOW_BUFFER_FT).unary_union],
                           crs=streams_clip.crs)
print(f"\nStep 4: Riparian buffer — {rip_buf.area.iloc[0] / SQFT_PER_ACRE:,.0f} acres")

# ═════════════════════════════════════════════════════════════
# Step 5: Overlay — parcel area inside the buffer, labelled with its watershed
# ArcPy: arcpy.analysis.Intersect([parcels, rip_buf, target_ws], rip_parcels)
# ═════════════════════════════════════════════════════════════
rip_parcels = gpd.overlay(parcels[['parcel_key', 'OWNER_NAME', 'OWNER_TYPE', 'area_acres', 'geometry']],
                          rip_buf, how='intersection', keep_geom_type=True)
rip_parcels = gpd.overlay(rip_parcels, target_ws[['WATERSHED_ID', 'WATERSHED_NAME', 'geometry']],
                          how='intersection', keep_geom_type=True)
rip_parcels['riparian_acres'] = rip_parcels.area / SQFT_PER_ACRE
rip_parcels['pct_of_parcel']  = (100 * rip_parcels['riparian_acres'] / rip_parcels['area_acres']).round(1)
print(f"\nStep 5: {rip_parcels['parcel_key'].nunique()} parcels have land in the buffer "
      f"({len(rip_parcels)} pieces after splitting by watershed)")

# ═════════════════════════════════════════════════════════════
# Step 6: Summarise
# ═════════════════════════════════════════════════════════════
summary = (rip_parcels.groupby(['WATERSHED_NAME', 'OWNER_TYPE'])
           .agg(n_parcels=('parcel_key', 'nunique'), riparian_acres=('riparian_acres', 'sum'))
           .round(2).sort_values('riparian_acres', ascending=False))
print(f"\nStep 6: Public riparian acres within {WORKFLOW_BUFFER_FT} ft of fish-bearing streams "
      f"({TARGET_SPECIES} watersheds):")
print(summary.to_string())

# ═════════════════════════════════════════════════════════════
# Step 7: Visualise the full pipeline result
# ═════════════════════════════════════════════════════════════
fig, ax = plt.subplots(figsize=(11, 9))
xmin, ymin, xmax, ymax = parcels.total_bounds
pad = 2000
target_ws.plot(ax=ax, color='#fff7bc', edgecolor='#cc4c02', linewidth=1)
rip_buf.plot(ax=ax, color='#9ecae1', alpha=0.6)
parcels.plot(ax=ax, facecolor='none', edgecolor='#969696', linewidth=0.3)
rip_parcels.plot(ax=ax, column='OWNER_TYPE', categorical=True, cmap='Dark2', legend=True,
                 legend_kwds={'title': 'Owner type (riparian part)', 'loc': 'lower left'})
streams_clip.plot(ax=ax, color='#08519c', linewidth=1)
ax.set_xlim(xmin - pad, xmax + pad); ax.set_ylim(ymin - pad, ymax + pad)
ax.set_title(f'Public parcel land within {WORKFLOW_BUFFER_FT} ft of fish-bearing streams\n'
             f'in watersheds with {TARGET_SPECIES} observations', fontsize=12, fontweight='bold')
ax.set_axis_off()
plt.tight_layout()
plt.savefig(OUT / 'm4_complete_workflow.png', dpi=150, bbox_inches='tight')
plt.show()

# ═════════════════════════════════════════════════════════════
# Step 8: Export — one GeoPackage, one layer per step, plus the summary table
# ArcPy: results already live in outputs/module4.gdb
# ═════════════════════════════════════════════════════════════
wf_gpkg = OUT / 'module4_workflow.gpkg'
if wf_gpkg.exists():
    wf_gpkg.unlink()
for lyr_name, gdf in {'target_watersheds': target_ws, 'fish_streams_clipped': streams_clip,
                      'riparian_buffer': rip_buf, 'riparian_parcels': rip_parcels}.items():
    gdf.to_file(wf_gpkg, layer=lyr_name, driver='GPKG')
summary.to_csv(OUT / 'module4_riparian_summary.csv')
print(f"\nOutputs saved:\n  {wf_gpkg}  →  {list_layers(wf_gpkg)}")
print(f"  {OUT / 'module4_riparian_summary.csv'}\n  {OUT / 'm4_complete_workflow.png'}")
print("\nNext step: git add . && git commit -m 'M4: riparian parcel workflow (sjoin-clip-buffer-overlay)'")

### 5b — Complete workflow in R

🔵 **`R`**

In [ ]:
# [R] Complete chained workflow: Spatial Join → Clip → Buffer → Overlay → Summarise → Export
TARGET_SPECIES     <- "Chinook"
WORKFLOW_BUFFER_FT <- 150

# Step 1: CRS check on the inputs read in Section 1
stopifnot(length(unique(sapply(list(watersheds, fish_streams, parcels, salmon_obs),
                               function(x) st_crs(x)$epsg))) == 1)

# Step 2: Spatial join — watersheds with the target species
# ArcPy: arcpy.analysis.SpatialJoin(obs, watersheds, obs_ws, match_option="WITHIN")
obs_ws_wf  <- st_join(salmon_obs, watersheds |> select(WATERSHED_ID), join = st_within, left = FALSE)
target_ids <- unique(obs_ws_wf$WATERSHED_ID[obs_ws_wf$species == TARGET_SPECIES])
target_ws  <- watersheds |> filter(WATERSHED_ID %in% target_ids)
cat("Step 2:", TARGET_SPECIES, "observed in", nrow(target_ws), "watersheds\n")

# Step 3: Clip fish-bearing streams to those watersheds (dissolve the mask first)
# ArcPy: arcpy.analysis.Clip(fish_streams, target_ws, streams_clip)
streams_clip <- st_intersection(fish_streams, st_union(target_ws))
cat("Step 3: Clipped streams —", nrow(streams_clip), "segments\n")

# Step 4: Buffer and dissolve
# ArcPy: arcpy.analysis.PairwiseBuffer(streams_clip, rip_buf, "150 Feet", dissolve_option="ALL")
rip_buf <- st_sf(buffer_ft = WORKFLOW_BUFFER_FT, geometry = st_union(st_buffer(streams_clip, WORKFLOW_BUFFER_FT)),
                 agr = "constant")
cat("Step 4: Riparian buffer —", round(as.numeric(st_area(rip_buf)) / SQFT_PER_ACRE), "acres\n")

# Step 5: Overlay — parcels ∩ buffer ∩ watersheds
# ArcPy: arcpy.analysis.Intersect([parcels, rip_buf, target_ws], rip_parcels)
rip_parcels <- parcels |>
  select(parcel_key, OWNER_NAME, OWNER_TYPE, area_acres) |>
  st_intersection(rip_buf) |>
  st_intersection(target_ws |> select(WATERSHED_ID, WATERSHED_NAME)) |>
  st_collection_extract("POLYGON") |>
  mutate(riparian_acres = as.numeric(st_area(geometry)) / SQFT_PER_ACRE)
cat("Step 5:", n_distinct(rip_parcels$parcel_key), "parcels have land in the buffer\n\n")

# Step 6: Summarise
summary_tbl <- rip_parcels |>
  st_drop_geometry() |>
  group_by(WATERSHED_NAME, OWNER_TYPE) |>
  summarise(n_parcels = n_distinct(parcel_key), riparian_acres = round(sum(riparian_acres), 2),
            .groups = "drop") |>
  arrange(desc(riparian_acres))
print(summary_tbl)

# Step 7: Visualise
bb <- st_bbox(parcels)
owner_cols <- c(CITY = "#1b9e77", COUNTY = "#d95f02", STATE = "#7570b3", SCHOOL = "#e7298a", FEDERAL = "#66a61e")
plot(st_geometry(target_ws), col = "#fff7bc", border = "#cc4c02",
     xlim = bb[c("xmin", "xmax")], ylim = bb[c("ymin", "ymax")],
     main = paste0("Public parcel land within ", WORKFLOW_BUFFER_FT, " ft of fish-bearing streams"))
plot(st_geometry(rip_buf), col = "#9ecae1", border = NA, add = TRUE)
plot(st_geometry(parcels), border = "#969696", lwd = 0.3, add = TRUE)
plot(st_geometry(rip_parcels), col = owner_cols[rip_parcels$OWNER_TYPE], border = NA, add = TRUE)
plot(st_geometry(streams_clip), col = "#08519c", lwd = 1, add = TRUE)
legend("bottomleft", legend = names(owner_cols), fill = owner_cols, title = "Owner type", cex = 0.8)

# Step 8: Export
wf_gpkg <- file.path(OUT, "module4_workflow_r.gpkg")
st_write(target_ws,    wf_gpkg, layer = "target_watersheds",    delete_dsn = TRUE, quiet = TRUE)
st_write(streams_clip, wf_gpkg, layer = "fish_streams_clipped", quiet = TRUE)
st_write(rip_buf,      wf_gpkg, layer = "riparian_buffer",      quiet = TRUE)
st_write(rip_parcels,  wf_gpkg, layer = "riparian_parcels",     quiet = TRUE)
write.csv(summary_tbl, file.path(OUT, "module4_riparian_summary_r.csv"), row.names = FALSE)
cat("\nOutputs saved:", wf_gpkg, "| module4_riparian_summary_r.csv\n")
cat("Next: git add . && git commit -m 'M4-R: riparian parcel workflow'\n")

---
## Section 6 — Common Errors and How to Fix Them

These three errors cause most spatial operation failures. Each one gives you an empty result or silently wrong geometry, which is worse than an error message. All three are easy to make with the Module 3 data.

| Error | Symptom | Fix |
|---|---|---|
| CRS mismatch | Empty (or wrong) result from a join or clip — maybe only a warning | Check `.crs` on both layers; reproject to match |
| Buffer in the wrong units | Wildly large polygons (degrees), or zones ~3× too small (feet vs. metres) | Reproject to a projected CRS; read its unit before choosing the number |
| Row multiplication | `len(joined) > len(left)` | Use a stricter predicate, join on a representative point, or keep the largest overlap |

🟢 **Python**

In [ ]:
# [Python] Error 1 — CRS mismatch: the partner agency GeoJSON (EPSG:4326) vs. watersheds (EPSG:2926)
partner = gpd.read_file(DATA / 'salmon_obs_partner_agency.geojson')
print(f"partner CRS:    EPSG:{partner.crs.to_epsg()}   e.g. x = {partner.geometry.x.iloc[0]:.4f}")
print(f"watersheds CRS: EPSG:{watersheds.crs.to_epsg()}   e.g. x = {watersheds.total_bounds[0]:,.0f}")

with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter('always')
    bad_join = gpd.sjoin(partner, watersheds, how='inner', predicate='within')
print(f"\nBAD join:  {len(bad_join)} of {len(partner)} points matched")
print(f"  warning raised: {[str(w.message)[:70] for w in caught] or 'none'}")
print("  → degrees (≈ -122) never fall inside polygons measured in feet (≈ 1,300,000). No error — just nothing.")

# ── Fix: reproject to the course CRS, then join ───────────────────────────
partner_2926 = partner.to_crs(COURSE_EPSG)
good_join = gpd.sjoin(partner_2926, watersheds[['WATERSHED_NAME', 'geometry']], how='left', predicate='within')
print(f"\nGOOD join: {good_join['WATERSHED_NAME'].notna().sum()} of {len(partner)} points matched; "
      f"{good_join['WATERSHED_NAME'].isna().sum()} fall outside every watershed (the two Puget Sound points from Module 3)")

In [ ]:
# [Python] Error 2 — Buffer in the wrong units: degrees, feet, and metres
pt_4326 = partner.iloc[[0]]                       # one partner observation, EPSG:4326 (degrees)
pt_2926 = pt_4326.to_crs(COURSE_EPSG)             # US survey feet
pt_utm  = pt_4326.to_crs(32610)                   # metres (UTM 10N)

with warnings.catch_warnings():
    warnings.simplefilter('ignore', UserWarning)  # geopandas warns about geographic buffers — that's the point
    bad_deg = pt_4326.buffer(150)                 # ❌ 150 degrees
bx = bad_deg.total_bounds                         # in degrees
ft_buf  = pt_2926.buffer(150)                     # 150 ft
m_buf   = pt_utm.buffer(150)                      # 150 m

print(f"buffer(150) in EPSG:4326 → 'radius' 150°: x {bx[0]:,.0f}…{bx[2]:,.0f}, y {bx[1]:,.0f}…{bx[3]:,.0f} "
      f"— wider than the whole planet (latitude stops at ±90)")
print(f"buffer(150) in EPSG:2926 → radius 150 ft ≈ 46 m     = {ft_buf.area.iloc[0] / SQFT_PER_ACRE:,.2f} acres")
print(f"buffer(150) in EPSG:32610 → radius 150 m ≈ 492 ft   = {m_buf.area.iloc[0] / 4046.856:,.2f} acres")
print("\nSame number, three answers. Read the CRS unit before you pick a distance:")
for epsg in (4326, 2926, 32610):
    print(f"  EPSG:{epsg:<6} unit = {gpd.GeoSeries([], crs=epsg).crs.axis_info[0].unit_name}")

# ── Visualise the feet-vs-metres trap (the degree buffer is too big to plot) ─
fig, ax = plt.subplots(figsize=(6, 6))
gpd.GeoSeries(m_buf, crs=32610).to_crs(COURSE_EPSG).plot(ax=ax, color='#fdd0a2', edgecolor='#e6550d', label='150 m')
gpd.GeoSeries(ft_buf, crs=COURSE_EPSG).plot(ax=ax, color='#9ecae1', edgecolor='#3182bd')
pt_2926.plot(ax=ax, color='black', markersize=15)
ax.set_title("buffer(150): metres (orange) vs. US feet (blue)", fontsize=11)
ax.set_axis_off()
plt.tight_layout()
plt.savefig(OUT / 'm4_buffer_units.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# [Python] Error 3 — Row multiplication: parcels that straddle a watershed boundary
ws_small = watersheds[['WATERSHED_ID', 'WATERSHED_NAME', 'geometry']]

joined_intersects = gpd.sjoin(parcels, ws_small, how='left', predicate='intersects')
print(f"Parcels:                    {len(parcels)} rows")
print(f"sjoin 'intersects':         {len(joined_intersects)} rows  ← duplicates!")
dups = joined_intersects[joined_intersects.index.duplicated(keep=False)]
print(f"  parcels matched to 2+ watersheds: {dups.index.nunique()}")
print(dups[['parcel_key', 'OWNER_NAME', 'WATERSHED_NAME']].head(6).to_string())

# Fix 1: stricter predicate — but parcels on the boundary now get NO watershed
joined_within = gpd.sjoin(parcels, ws_small, how='left', predicate='within')
print(f"\nFix 1 — 'within':           {len(joined_within)} rows, "
      f"{joined_within['WATERSHED_ID'].isna().sum()} parcels left without a watershed")

# Fix 2: join on a point guaranteed to be inside each parcel (ArcPy: HAVE_THEIR_CENTER_IN)
pts = parcels.copy()
pts['geometry'] = parcels.representative_point()
by_point = gpd.sjoin(pts, ws_small, how='left', predicate='within')
parcels_ws = parcels.join(by_point[['WATERSHED_ID', 'WATERSHED_NAME']])
print(f"Fix 2 — representative pt:  {len(parcels_ws)} rows, {parcels_ws['WATERSHED_ID'].isna().sum()} unmatched")

# Fix 3: keep the watershed with the largest overlap (most defensible for reporting)
pieces = gpd.overlay(parcels[['parcel_key', 'geometry']], ws_small, how='intersection', keep_geom_type=True)
pieces['overlap_sqft'] = pieces.area
largest = pieces.sort_values('overlap_sqft', ascending=False).drop_duplicates('parcel_key')
print(f"Fix 3 — largest overlap:    {len(largest)} rows (one per parcel)")

agree = (parcels_ws.set_index('parcel_key')['WATERSHED_ID']
         .eq(largest.set_index('parcel_key')['WATERSHED_ID'])).mean()
print(f"\nFix 2 and Fix 3 agree for {agree:.0%} of parcels.")

---
## Section 7 — Extra Credit: Your Turn

**Scenario:** The watershed team wants a quick summer water-temperature screen along the **mainstem salmon corridors**: the Cedar River, Issaquah Creek, Bear Creek and the Sammamish River. Which of the 800 water-quality sites sit within **300 ft** of those streams, and what's the average temperature by watershed?

| Input | Source |
|---|---|
| Streams | `salmon_watersheds.gdb` → `streams` |
| Watersheds | `salmon_watersheds.gdb` → `watersheds` |
| Water-quality sites | `monitoring_sites.csv` (lon/lat — promote to points as in Module 3) |

Work through the steps below in order with your breakout group. Complete steps 1–4 before moving to step 5. After each step, **plot the output** to check it before you move on.

**Deliverable:** `outputs/module4_lab_output.gpkg` (layers `corridor_buffer`, `corridor_sites`) and `outputs/module4_lab_summary.csv`, with each step producing printed output or a plot that confirms it worked.

*Want to use your own data instead?* Swap the layers in Step 1 for two of your own (one polygon, one point or line) and keep the same five steps.

🟢 **Python**

In [ ]:
# [Python] Lab Step 1: Load & Inspect — read from the geodatabase and the CSV, check CRS for each
def sanity_check(gdf, label):
    print(f"── {label} ──")
    print(f"  Features: {len(gdf)} | Columns: {len(gdf.columns)}")
    print(f"  Geometry: {list(gdf.geom_type.unique())}")
    print(f"  CRS:      EPSG:{gdf.crs.to_epsg()} ({gdf.crs.axis_info[0].unit_name})")
    print(f"  Null/empty geometries: {int(gdf.geometry.isna().sum() + gdf.geometry.is_empty.sum())}")

lab_streams    = gpd.read_file(DATA / 'salmon_watersheds.gdb', layer='streams')
lab_watersheds = gpd.read_file(DATA / 'salmon_watersheds.gdb', layer='watersheds')
ms = pd.read_csv(DATA / 'monitoring_sites.csv')
lab_sites = gpd.GeoDataFrame(ms, geometry=gpd.points_from_xy(ms['longitude'], ms['latitude']), crs=4326)

for gdf, label in [(lab_streams, 'Streams (GDB)'), (lab_watersheds, 'Watersheds (GDB)'), (lab_sites, 'Sites (CSV)')]:
    sanity_check(gdf, label)

# ── Reproject to the shared course CRS ────────────────────────────────────
lab_sites = lab_sites.to_crs(COURSE_EPSG)
print(f"\nSites reprojected → EPSG:{lab_sites.crs.to_epsg()}. All three layers now share the course CRS.")

In [ ]:
# [Python] Lab Step 2: Buffer — 300 ft around the mainstem corridors
CORRIDORS = ['Cedar River', 'Issaquah Creek', 'Bear Creek', 'Sammamish River']
LAB_BUFFER_FT = 300

# ── YOUR CODE HERE ────────────────────────────────────────────
# corridors = lab_streams[lab_streams['STREAM_NAME'].isin(CORRIDORS)]
# corridor_buffer = gpd.GeoDataFrame(geometry=corridors.buffer(LAB_BUFFER_FT), crs=corridors.crs).dissolve()
# corridor_buffer['buffer_ft'] = LAB_BUFFER_FT
# print(f"{len(corridors)} corridor segments → buffer of {corridor_buffer.area.iloc[0] / SQFT_PER_ACRE:,.0f} acres")

# ── Verify: plot the result ───────────────────────────────────
# ax = lab_watersheds.plot(color='#f2f2f2', edgecolor='#999999')
# corridor_buffer.plot(ax=ax, color='#9ecae1')
# corridors.plot(ax=ax, color='#08519c', linewidth=0.8)

print("Complete Lab Step 2 — Buffer")
print("Expected: one dissolved polygon, EPSG:2926, a few thousand acres. Units are FEET.")

In [ ]:
# [Python] Lab Step 3: Clip — keep only the sites inside the corridor buffer

# ── YOUR CODE HERE ────────────────────────────────────────────
# corridor_sites = gpd.clip(lab_sites, corridor_buffer)
# print(f"Sites before clip: {len(lab_sites)} | after clip: {len(corridor_sites)}")

# ── Verify ───────────────────────────────────────────────────
# ax = corridor_buffer.plot(color='#deebf7')
# lab_sites.plot(ax=ax, color='#bdbdbd', markersize=3)
# corridor_sites.plot(ax=ax, color='#d94801', markersize=8)

print("Complete Lab Step 3 — Clip")
print("Expected: well under 800 sites remain (roughly 140 — about one in six).")

In [ ]:
# [Python] Lab Step 4: Spatial Join — which watershed is each corridor site in?
# Start with predicate='within'. Check row counts before and after.

# ── YOUR CODE HERE ────────────────────────────────────────────
# corridor_sites = gpd.sjoin(
#     corridor_sites,                                          # left: points
#     lab_watersheds[['WATERSHED_ID', 'WATERSHED_NAME', 'geometry']],  # right: polygons
#     how='left',
#     predicate='within'
# ).drop(columns='index_right')
# print(f"Points before: {len(gpd.clip(lab_sites, corridor_buffer))} | after join: {len(corridor_sites)}")
# print(f"Unmatched: {corridor_sites['WATERSHED_ID'].isna().sum()}")

print("Complete Lab Step 4 — Spatial Join")
print("Expected: same row count as Step 3, with WATERSHED_ID and WATERSHED_NAME added.")

In [ ]:
# [Python] Lab Step 5: Summarise & Export
# Mean water temperature by watershed; export the layers to one GeoPackage and the table to CSV.

# ── YOUR CODE HERE ────────────────────────────────────────────
# lab_summary = (corridor_sites.groupby('WATERSHED_NAME')
#                .agg(n_sites=('site_id', 'count'), mean_temp_c=('value', 'mean'), max_temp_c=('value', 'max'))
#                .round(1).sort_values('mean_temp_c', ascending=False))
# print(lab_summary)

# lab_gpkg = OUT / 'module4_lab_output.gpkg'
# corridor_buffer.to_file(lab_gpkg, layer='corridor_buffer', driver='GPKG')
# corridor_sites.to_file(lab_gpkg, layer='corridor_sites', driver='GPKG')
# lab_summary.to_csv(OUT / 'module4_lab_summary.csv')
# print(f"Saved: {lab_gpkg} → {list_layers(lab_gpkg)}")

print("Complete Lab Step 5 — Summarise & Export")
print("Expected: a summary table — which watersheds run warmest, and why might the upper Cedar be coolest?")

**Lab Steps — R version.** The same five steps in R. Run the R cell in **Section 1** first, then fill in the `# TODO` lines.

🔵 **`R`**

In [ ]:
# [R] Lab Steps 1-5 — corridor water-temperature screen, R version
CORRIDORS     <- c("Cedar River", "Issaquah Creek", "Bear Creek", "Sammamish River")
LAB_BUFFER_FT <- 300

# Step 1 — Load & Inspect
lab_streams    <- st_read(file.path(DATA, "salmon_watersheds.gdb"), layer = "streams",    quiet = TRUE)
lab_watersheds <- st_read(file.path(DATA, "salmon_watersheds.gdb"), layer = "watersheds", quiet = TRUE)
lab_sites <- read.csv(file.path(DATA, "monitoring_sites.csv")) |>
  st_as_sf(coords = c("longitude", "latitude"), crs = 4326) |>
  st_transform(COURSE_EPSG)
st_agr(lab_sites) <- "constant"
cat("streams:", nrow(lab_streams), "| watersheds:", nrow(lab_watersheds), "| sites:", nrow(lab_sites),
    "| all EPSG", st_crs(lab_sites)$epsg, "\n")

# Step 2 — Buffer   (TODO: filter to CORRIDORS, st_buffer by LAB_BUFFER_FT, dissolve with st_union)
# corridor_buffer <- lab_streams |> filter(STREAM_NAME %in% CORRIDORS) |>
#   st_buffer(LAB_BUFFER_FT) |> st_union() |> st_sf(buffer_ft = LAB_BUFFER_FT, geometry = _)

# Step 3 — Clip     (TODO: keep sites inside the buffer)
# corridor_sites <- st_intersection(lab_sites, st_geometry(corridor_buffer))

# Step 4 — Spatial join (TODO: add WATERSHED_NAME with st_within; check nrow before/after)
# corridor_sites <- st_join(corridor_sites, lab_watersheds |> select(WATERSHED_ID, WATERSHED_NAME), join = st_within)

# Step 5 — Summarise & export
# lab_summary <- corridor_sites |> st_drop_geometry() |> group_by(WATERSHED_NAME) |>
#   summarise(n_sites = n(), mean_temp_c = round(mean(value), 1)) |> arrange(desc(mean_temp_c))
# st_write(corridor_buffer, file.path(OUT, "module4_lab_output_r.gpkg"), layer = "corridor_buffer", delete_dsn = TRUE)
# st_write(corridor_sites,  file.path(OUT, "module4_lab_output_r.gpkg"), layer = "corridor_sites")
# write.csv(lab_summary, file.path(OUT, "module4_lab_summary_r.csv"), row.names = FALSE)

---
## Section 8 — Extra Credit Challenge Extensions

Pick any one of these if you finish the guided lab early or want to push further. Each maps to the Extended Application options in the session.

### Option A — Predicate Exploration
Join **parcels** to **watersheds** with `within`, `intersects`, `contains` and `gpd.sjoin_nearest()`. Then do the same with the **salmon observations**. Print the row count and a sample of results for each. Write three sentences explaining why the counts differ between the two layers.

### Option B — Parameterised Workflow Function
Wrap the Section 5 workflow in a function that takes `buffer_ft` and `species` as parameters. Run it for 100, 150 and 300 ft and for each species. How sensitive is the riparian-acres answer to the buffer rule? (Starter below.)

### Option C — Your Own Data
Apply the complete workflow to a dataset from your own professional work. Document:
- What question you're answering
- What CRS you chose, what its units are, and why
- What the result means for your work

### Option D — R + Python (+ ArcPy) Cross-check
Run the Section 5 workflow in Python and R (and ArcPy if you have Pro) on the same GDB inputs. Compare the feature counts, column names and riparian acres. What matches? What differs, and why? (Look at how each tool handles slivers and geometry collections.)

🟢 **Python**

In [ ]:
# [Python] Option B starter: parameterised riparian-parcel workflow
def riparian_parcels_summary(buffer_ft, species='Chinook',
                             obs=salmon_obs, ws=watersheds, streams_=fish_streams, parcels_=parcels):
    '''
    Spatial join → clip → buffer → overlay → summarise, as in Section 5.

    Parameters
    ----------
    buffer_ft : float — riparian buffer distance in US survey feet (EPSG:2926 units)
    species   : str   — salmon species that defines the target watersheds

    Returns
    -------
    dict with the target watershed count, buffer acres, riparian acres and a summary table
    '''
    assert len({g.crs.to_epsg() for g in (obs, ws, streams_, parcels_)}) == 1, "CRS mismatch"
    hits = gpd.sjoin(obs[obs['species'] == species], ws[['WATERSHED_ID', 'geometry']], predicate='within')
    target = ws[ws['WATERSHED_ID'].isin(hits['WATERSHED_ID'])]
    zone = gpd.clip(streams_, target).buffer(buffer_ft)
    zone = gpd.GeoDataFrame(geometry=[zone.union_all() if hasattr(zone, 'union_all') else zone.unary_union],
                            crs=ws.crs)
    rip = gpd.overlay(parcels_[['parcel_key', 'OWNER_TYPE', 'geometry']], zone,
                      how='intersection', keep_geom_type=True)
    rip['riparian_acres'] = rip.area / SQFT_PER_ACRE
    return {
        'buffer_ft': buffer_ft, 'species': species,
        'n_watersheds': len(target),
        'buffer_acres': zone.area.sum() / SQFT_PER_ACRE,
        'riparian_acres': rip['riparian_acres'].sum(),
        'summary': rip.groupby('OWNER_TYPE')['riparian_acres'].sum().round(2),
    }

# ── Run for three distances ───────────────────────────────────
rows = []
for dist in [100, 150, 300]:
    r = riparian_parcels_summary(dist)
    rows.append({k: r[k] for k in ('buffer_ft', 'n_watersheds', 'buffer_acres', 'riparian_acres')})
    print(f"\n── {dist} ft ── riparian public acres by owner type:\n{r['summary'].to_string()}")
print("\n", pd.DataFrame(rows).round(1).to_string(index=False))

# Next: loop over species too — does the Sockeye answer differ from Chinook? Why?

---
### 🍍 Self-Check Questions

1. Why does `buffer(150)` mean something different on `fish_streams` (EPSG:2926) than on a layer in EPSG:32610? How do you check a CRS's unit in code?
2. You need each parcel's riparian acres **and** its owner type. Why does `gpd.overlay(how='intersection')` fit better than `gpd.clip()`?
3. The parcels → watersheds join with `intersects` returned more rows than there are parcels. Give two ways to fix that, and say when you'd prefer each.
4. In Section 6, the CRS-mismatch join returned zero rows. Did it raise an error? What habit catches this before it reaches a report?

---
## ✅ Module 4 Homework — Deliverables

Submit **two notebooks with output** 

### Notebook 1 — Python
Show:
- Python code and output for sections 1-5
- Code runs successfully and attempt 'Try it Yourself' in new cells
- Sections 7-8 are extra credit

### Notebook 2 — R
Show:
- R code and output for sections 1-5
- Code runs successfully and attempt 'Try it Yourself' in new cells
- Sections 7-8 are extra credit

---

**If you want to use git and GitHub to push work to your GitHub repo**
```
git add GISPR_Module4_VectorOperations.ipynb outputs/module4_*.gpkg outputs/module4_*.csv outputs/m4_*.png
git commit -m "M4: buffer, clip, sjoin complete — all steps documented"
git push
```
Put link to your GitHub repo in notebook or better yet put link in Assignment submission

**Stuck?** Post in Ed Discussion (tag `#module4`) with:
- The exact error message (screenshot or copy-paste)
- Which cell failed
- The CRS **and unit** of each layer you were working with

If you solved an error, share how — someone else is probably hitting the same thing.

---

## What's Next

**Module 5 — Introduction to Raster Data.** From vectors to grids: data models, arrays, file formats, and loading rasters with `rasterio`, `terra` and `stars`. The watershed polygons you joined tonight come back in Module 6 as zones for zonal statistics.